**TEACHER FORCING IMPLEMENTATION**

Write a custom training loop for the Seq2Seq model that utilizes a randomized Teacher forcing ratio during decoder prediction.

In [1]:
import os
import re
import random
import zipfile
import requests

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence


In [5]:
# REPRODUCIBILITY

random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


# Select CPU or GPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


Using device: cuda


In [6]:
# DOWNLOAD MANYTHINGS ENGLISH-FRENCH DATASET

URL = "https://www.manythings.org/anki/fra-eng.zip"
ZIP_FILE = "fra-eng.zip"
TXT_FILE = "fra.txt"

if not os.path.exists(TXT_FILE):
    print("\nDownloading ManyThings English-French dataset...")

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5"
    }

    response = requests.get(URL, headers=headers, stream=True)
    response.raise_for_status()

    with open(ZIP_FILE, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

    print("Download completed.")

    # Extract ZIP file
    with zipfile.ZipFile(ZIP_FILE, "r") as zip_ref:
        zip_ref.extractall(".")

    print("Dataset extracted.")
else:
    print("\nDataset already exists.")
    print("Skipping download.")



Dataset already exists.
Skipping download.


In [7]:
# TEXT CLEANING

def clean_text(text):

    # Convert to lowercase
    text = text.lower().strip()

    # Separate punctuation from words
    text = re.sub(
        r"([.!?])",
        r" \1",
        text
    )

    # Keep English/French letters and punctuation
    text = re.sub(
        r"[^a-zA-ZÀ-ÿ.!?]+",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


In [8]:
# LOAD A SUBSET OF MANYTHINGS DATA

MAX_PAIRS = 5000

raw_pairs = []

with open(
    TXT_FILE,
    "r",
    encoding="utf-8"
) as f:

    for i, line in enumerate(f):

        # Stop after selected number of examples
        if i >= MAX_PAIRS:
            break

        parts = line.split("\t")

        if len(parts) >= 2:

            english = clean_text(parts[0])
            french = clean_text(parts[1])

            # Ignore empty sentences
            if english and french:

                raw_pairs.append(
                    (english, french)
                )


print("\nDataset loaded.")
print(
    "Number of sentence pairs:",
    len(raw_pairs)
)


Dataset loaded.
Number of sentence pairs: 5000


In [9]:
# VOCABULARY CLASS

class Vocabulary:

    def __init__(self):

        # Special tokens
        self.word2idx = {

            "<PAD>": 0,
            "<SOS>": 1,
            "<EOS>": 2,
            "<UNK>": 3

        }

        self.idx2word = {

            0: "<PAD>",
            1: "<SOS>",
            2: "<EOS>",
            3: "<UNK>"

        }

        self.num_words = 4


    def add_sentence(self, sentence):

        for word in sentence.split():

            if word not in self.word2idx:

                self.word2idx[word] = self.num_words

                self.idx2word[
                    self.num_words
                ] = word

                self.num_words += 1


    def numericalize(self, sentence):

        tokens = [

            self.word2idx["<SOS>"]

        ]

        for word in sentence.split():

            tokens.append(

                self.word2idx.get(
                    word,
                    self.word2idx["<UNK>"]
                )

            )

        tokens.append(

            self.word2idx["<EOS>"]

        )

        return tokens

In [10]:
# BUILD SOURCE AND TARGET VOCABULARIES


src_vocab = Vocabulary()
trg_vocab = Vocabulary()

for english, french in raw_pairs:

    src_vocab.add_sentence(
        english
    )

    trg_vocab.add_sentence(
        french
    )


print("\nVocabulary Information")

print(
    "English vocabulary:",
    src_vocab.num_words
)

print(
    "French vocabulary:",
    trg_vocab.num_words
)



Vocabulary Information
English vocabulary: 1084
French vocabulary: 2270


In [11]:
# TRANSLATION DATASET

class TranslationDataset(Dataset):

    def __init__(
         self,
         pairs,
         src_vocab,
         trg_vocab
    ):

         self.pairs = pairs
         self.src_vocab = src_vocab
         self.trg_vocab = trg_vocab

    def __len__(self):
         return len(self.pairs)

    def __getitem__(self, index):
         english, french = self.pairs[index]
         src_tensor = torch.tensor(
             self.src_vocab.numericalize(
                 english
             ),
             dtype=torch.long
         )
         trg_tensor = torch.tensor(
             self.trg_vocab.numericalize(
                 french
             ),
             dtype=torch.long
         )
         return src_tensor, trg_tensor

In [12]:
# 9. DYNAMIC PADDING
def pad_collate(batch):

    src_batch, trg_batch = zip(*batch)

    # Pad source sentences
    src_batch = pad_sequence(
        src_batch,
        batch_first=True,
        padding_value=0
    )

    # Pad target sentences
    trg_batch = pad_sequence(
        trg_batch,
        batch_first=True,
        padding_value=0
    )

    return src_batch, trg_batch


In [13]:
# CREATE DATASET AND DATALOADER

dataset = TranslationDataset(
    raw_pairs,
    src_vocab,
    trg_vocab
)

dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=pad_collate
)


In [14]:
# LSTM ENCODER
class Encoder(nn.Module):

    def __init__(
        self,
        input_dim,
        emb_dim,
        hidden_dim
    ):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell


In [15]:
# LSTM DECODER

class Decoder(nn.Module):

    def __init__(
        self,
        output_dim,
        emb_dim,
        hidden_dim
    ):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, input_token, hidden, cell):
        input_token = input_token.unsqueeze(1)
        embedded = self.embedding(input_token)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell

In [16]:
# SEQ2SEQ MODEL

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder



In [17]:
# CREATE MODEL

INPUT_DIM = src_vocab.num_words
OUTPUT_DIM = trg_vocab.num_words
EMB_DIM = 64
HID_DIM = 128

encoder = Encoder(INPUT_DIM, EMB_DIM, HID_DIM)
decoder = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM)
model = Seq2Seq(encoder, decoder).to(device)


In [18]:
# LOSS AND OPTIMIZER

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [19]:
# TEACHER FORCING TRAINING FUNCTION

def train_epoch(
    model,
    dataloader,
    optimizer,
    criterion,
    teacher_forcing_ratio,
    device
):
    model.train()
    epoch_loss = 0
    teacher_forced_steps = 0
    model_prediction_steps = 0

    for src, trg in dataloader:
        src = src.to(device)
        trg = trg.to(device)
        optimizer.zero_grad()

        hidden, cell = model.encoder(src)
        decoder_input = trg[:, 0]
        loss = 0
        trg_len = trg.shape[1]

        for t in range(1, trg_len):
            output, hidden, cell = model.decoder(decoder_input, hidden, cell)
            loss += criterion(output, trg[:, t])

            use_teacher_forcing = random.random() < teacher_forcing_ratio

            if use_teacher_forcing:
                decoder_input = trg[:, t]
                teacher_forced_steps += 1
            else:
                decoder_input = output.argmax(dim=1)
                model_prediction_steps += 1

        loss = loss / (trg_len - 1)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    total_steps = teacher_forced_steps + model_prediction_steps
    observed_tf_ratio = teacher_forced_steps / total_steps if total_steps > 0 else 0

    print(f"Teacher-forced steps     : {teacher_forced_steps}")
    print(f"Model-predicted steps    : {model_prediction_steps}")
    print(f"Observed TF ratio        : {observed_tf_ratio:.3f}")

    return epoch_loss / len(dataloader)

In [20]:
# STANDALONE INFERENCE FUNCTION

def translate_sentence(
    model,
    sentence,
    src_vocab,
    trg_vocab,
    max_len=20
):
    model.eval()
    src_tokens = src_vocab.numericalize(sentence)
    src_tensor = torch.tensor(src_tokens, dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    decoder_input = torch.tensor([trg_vocab.word2idx["<SOS>"]], dtype=torch.long).to(device)
    predicted_words = []

    for _ in range(max_len):
        with torch.no_grad():
            output, hidden, cell = model.decoder(decoder_input, hidden, cell)

        predicted_id = output.argmax(dim=1).item()
        if predicted_id == trg_vocab.word2idx["<EOS>"]:
            break

        if predicted_id not in [trg_vocab.word2idx["<PAD>"], trg_vocab.word2idx["<SOS>"]]:
            predicted_words.append(trg_vocab.idx2word[predicted_id])

        decoder_input = torch.tensor([predicted_id], dtype=torch.long).to(device)

    return " ".join(predicted_words)

In [21]:
# FUNCTION TO TEST TRANSLATIONS

def show_translations(
    model,
    test_pairs,
    src_vocab,
    trg_vocab,
    title
):
    print("\n")
    print("=" * 70)
    print(title)
    print("=" * 70)

    for english, target in test_pairs:
        prediction = translate_sentence(model, english, src_vocab, trg_vocab)
        print(f"English   : {english}")
        print(f"Target    : {target}")
        print(f"Predicted : {prediction}")
        print("-" * 70)


In [22]:
# SELECT TEST SENTENCES

test_pairs = raw_pairs[:5]

In [24]:
# REDUCING TEACHER FORCING EXPERIMENT

teacher_forcing_schedule = [1.00, 0.75, 0.50, 0.25, 0.00]
EPOCHS_PER_STAGE = 2

for ratio in teacher_forcing_schedule:
    print("\n\n")
    print("=" * 70)
    print(f"TRAINING WITH TEACHER FORCING RATIO = {ratio:.2f}")
    print("=" * 70)

    for epoch in range(1, EPOCHS_PER_STAGE + 1):
        loss = train_epoch(
            model,
            dataloader,
            optimizer,
            criterion,
            teacher_forcing_ratio=ratio,
            device=device
        )
        print(f"Ratio: {ratio:.2f} | Epoch: {epoch} | Loss: {loss:.4f}")

    show_translations(
        model,
        test_pairs,
        src_vocab,
        trg_vocab,
        title=f"INFERENCE AFTER TRAINING WITH TF RATIO = {ratio:.2f}"
    )




TRAINING WITH TEACHER FORCING RATIO = 1.00
Teacher-forced steps     : 1274
Model-predicted steps    : 0
Observed TF ratio        : 1.000
Ratio: 1.00 | Epoch: 1 | Loss: 2.0042
Teacher-forced steps     : 1291
Model-predicted steps    : 0
Observed TF ratio        : 1.000
Ratio: 1.00 | Epoch: 2 | Loss: 1.8477


INFERENCE AFTER TRAINING WITH TF RATIO = 1.00
English   : go .
Target    : va !
Predicted : soyez !
----------------------------------------------------------------------
English   : go .
Target    : marche .
Predicted : soyez !
----------------------------------------------------------------------
English   : go .
Target    : en route !
Predicted : soyez !
----------------------------------------------------------------------
English   : go .
Target    : bouge !
Predicted : soyez !
----------------------------------------------------------------------
English   : hi .
Target    : salut !
Predicted : va toi .
----------------------------------------------------------------------


In [25]:
# FINAL FULLY AUTOREGRESSIVE INFERENCE

print("\n")
print("=" * 70)
print("FINAL INFERENCE — NO TEACHER FORCING")
print("=" * 70)

for english, target in test_pairs:
    prediction = translate_sentence(model, english, src_vocab, trg_vocab)
    print(f"\nEnglish   : {english}")
    print(f"Target    : {target}")
    print(f"Predicted : {prediction}")



FINAL INFERENCE — NO TEACHER FORCING

English   : go .
Target    : va !
Predicted : allez .

English   : go .
Target    : marche .
Predicted : allez .

English   : go .
Target    : en route !
Predicted : allez .

English   : go .
Target    : bouge !
Predicted : allez .

English   : hi .
Target    : salut !
Predicted : va .
